<a href="https://colab.research.google.com/github/GaganaGB/WaterConsumptionAnomalyClassification/blob/main/Water_Consumption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q pandas numpy scikit-learn streamlit joblib

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import joblib

# 1. Synthetic Data Generation
np.random.seed(42)
n_samples = 1500

hour = np.random.randint(0, 24, n_samples)
occupancy = np.random.randint(1, 6, n_samples)
temperature = np.random.normal(25, 5, n_samples)
historical_usage = np.random.uniform(10, 50, n_samples)

# Normal consumption based on baseline and context
consumption = historical_usage * (1 + 0.05 * occupancy) + np.random.normal(0, 3, n_samples)

# Inject anomalies (~10% rate)
anomaly_idx = np.random.choice(n_samples, size=int(n_samples * 0.10), replace=False)
consumption[anomaly_idx] += np.random.uniform(40, 120, len(anomaly_idx))

is_anomaly = np.zeros(n_samples, dtype=int)
is_anomaly[anomaly_idx] = 1

df = pd.DataFrame({
    'hour': hour,
        'occupancy': occupancy,
            'temperature': temperature,
                'historical_usage': historical_usage,
                    'consumption': consumption,
                        'is_anomaly': is_anomaly
                        })

                        # 2. Feature Engineering
df['usage_ratio'] = df['consumption'] / (df['historical_usage'] + 1e-5)
df['per_capita_usage'] = df['consumption'] / df['occupancy']
df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24.0)

X = df.drop(columns=['is_anomaly'])
y = df['is_anomaly']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

                            # 3. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

                            # 4. Model Training & Metrics
models = {"Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42),"KNN": KNeighborsClassifier(n_neighbors=5),"Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)}

print("=== MODEL EVALUATION METRICS ===")
for name, model in models.items():
  model.fit(X_train_scaled, y_train)
  preds = model.predict(X_test_scaled)
  probs = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else None

  print(f"\nModel: {name}")
  print(classification_report(y_test, preds, target_names=["Normal", "Anomaly"]))
  if probs is not None:
    print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")

                                                                            # Save artifacts
best_model = models["Decision Tree"]
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(best_model, 'model.joblib')
print("\nTrained model and scaler successfully saved!")

=== MODEL EVALUATION METRICS ===

Model: Logistic Regression
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       270
     Anomaly       1.00      1.00      1.00        30

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

ROC-AUC: 1.0000

Model: KNN
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       270
     Anomaly       1.00      0.97      0.98        30

    accuracy                           1.00       300
   macro avg       1.00      0.98      0.99       300
weighted avg       1.00      1.00      1.00       300

ROC-AUC: 1.0000

Model: Decision Tree
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       270
     Anomaly       0.97      1.00      0.98        30

    accuracy                           1.00       300
 

In [15]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

@st.cache_resource
def load_artifacts():
    scaler = joblib.load('scaler.joblib')
    model = joblib.load('model.joblib')
    return scaler, model
scaler, model = load_artifacts()

st.title("Smart Water Anomaly Detector")
st.caption("Contextual Water Consumption Anomaly Classification Prototype")

st.sidebar.header("Input Meter Data")
hour = st.sidebar.slider("Hour of Day", 0, 23, 14)
occupancy = st.sidebar.number_input("Occupancy Count", min_value=1, max_value=10, value=3)
temperature = st.sidebar.slider("Ambient Temperature (°C)", -5.0, 45.0, 28.0)
historical_usage = st.sidebar.number_input("Historical Usage (L)", min_value=1.0, value=30.0)
consumption = st.sidebar.number_input("Current Meter Reading (L)", min_value=0.0, value=35.0)

input_data = pd.DataFrame([{
    'hour': hour,
        'occupancy': occupancy,
            'temperature': temperature,
                'historical_usage': historical_usage,
                    'consumption': consumption,
                        'usage_ratio': consumption / (historical_usage + 1e-5),
                            'per_capita_usage': consumption / occupancy,
                                'sin_hour': np.sin(2 * np.pi * hour / 24.0),
                                    'cos_hour': np.cos(2 * np.pi * hour / 24.0)
                                    }])
scaled_input = scaler.transform(input_data)

if st.button("Classify Reading"):
  prediction = model.predict(scaled_input)[0]
  probabilities = model.predict_proba(scaled_input)[0]
  risk_score = probabilities[1] * 100
  col1, col2 = st.columns(2)
      with col1:
        if prediction == 1:
          st.error("🚨 **Classification: ANOMALY DETECTED**")
          st.write("Unusual consumption relative to historical baseline and context.")
        else:
          st.success("✅ **Classification: NORMAL CONSUMPTION**")
          st.write("Usage is within expected thresholds.")

      with col2:
        st.subheader(f"Risk Score: {risk_score:.1f}%")
        st.progress(int(risk_score))


Overwriting app.py


In [16]:
import subprocess
import time

# 1. Start Streamlit app in the background
subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(3)

# 2. Print your Colab IP address (Needed as the password)
print("👉 YOUR LOCALTUNNEL PASSWORD (IP Address) IS:")
!curl -s https://ipv4.icanhazip.com

# 3. Create the public tunnel
print("\n🔗 CLICK THE LINK BELOW TO OPEN YOUR APP:")
!npx localtunnel --port 8501

👉 YOUR LOCALTUNNEL PASSWORD (IP Address) IS:
34.16.23.115

🔗 CLICK THE LINK BELOW TO OPEN YOUR APP:
⠙⠹⠸⠼⠴⠦your url is: https://fuzzy-candles-add.loca.lt
^C
